# 02 – Análisis

**Proyecto:** Evolución de los ingresos laborales según edad, educación y región en Argentina: antes, durante y después de la cuarentena.
**Fuente:** Encuesta Permanente de Hogares (EPH) del INDEC, 2.º trimestre de 2017 a 2026.
**Población:** personas de 18 años o más de los 31 aglomerados urbanos de la EPH.
**Unidad de los ingresos:** pesos de octubre de 2026.
**Etapa del proyecto:** Etapa 4 – Analizar.

### Objetivo
Calcular estadísticos ponderados del ingreso laboral real (media, mediana y dispersión) y las tasas de empleo y desempleo; compararlos por año, edad, nivel educativo y región; resumirlos en las tres etapas (antes, durante y después de la cuarentena), y comprobar la solidez de los resultados con controles, una validación externa y un análisis de sensibilidad.

### Entradas
| Archivo | Contenido |
|---|---|
| `datos_procesados/eph_t2_2017_2026_18mas_ingresos.parquet` | Base de adultos con ingresos reales, generada por `01_preparacion` |
| `datos_originales/ipc/serie_ipc_divisiones.csv` | Serie del IPC del INDEC. Se usa solo en la sección 13 |

### Salidas (carpeta `resultados/`)
| Archivo | Contenido |
|---|---|
| `p21_real_anual.csv` | Indicador central por año: casos, media, percentiles y dispersión |
| `ing_lab_real_anual.csv` | Indicador ampliado por año |
| `p21_real_anual_edad.csv` | Indicador central por año y grupo de edad, con su precisión |
| `p21_real_anual_educacion.csv` | Indicador central por año y nivel educativo, con su precisión |
| `p21_real_anual_region.csv` | Indicador central por año y región, con su precisión |
| `robustez_variaciones.csv` | Variación 2017–2026 con media y mediana, y si es robusta |
| `tasas_anual.csv` | Tasas de empleo y desempleo, y proporción de ocupados con ingreso cero |
| `resumen_etapas.csv` | Promedio por etapa de los indicadores generales |
| `resumen_etapas_edad.csv`, `resumen_etapas_educacion.csv`, `resumen_etapas_region.csv` | Media y mediana por etapa y grupo |
| `correlacion_educ_ingreso_anual.csv` | Correlación de Spearman ponderada entre educación e ingreso, por año |
| `sensibilidad_deflactor.csv` | Media anual del indicador central con cuatro criterios de deflactor |

### Orden de ejecución
Se ejecuta después de `01_preparacion` y antes de `03_visualizaciones`. Si la sesión de Colab se reinicia, hay que volver a ejecutar las secciones 1 y 2 antes de continuar.

### Criterios del análisis
- **Ponderadores:** PONDIIO para los ingresos (`p21_real` e `ing_lab_real`); PONDERA para las tasas y para la proporción de ocupados con ingreso cero.
- **Percentiles ponderados:** el primer valor cuyo peso acumulado alcanza el corte, sin interpolar (equivale a `method='inverted_cdf'` de numpy).
- **Dispersión:** el rango intercuartílico (RIQ) es la medida principal. El desvío estándar y el cociente p90/p10 la complementan.
- **Valores extremos:** no se recortan; se informa su efecto (sección 7.1).
- **Precisión de los cruces:** con 100 casos o más se publica; entre 30 y 99, con advertencia; con menos de 30, no se publica.
- **Robustez:** una variación es robusta si la media y la mediana coinciden en la dirección y difieren en menos de 4 puntos. Si no, se informa con la media.
- **Resumen por etapa:** promedio simple de los valores anuales, de modo que cada año pesa igual.
- **Alcance:** los resultados son descriptivos. Ninguna relación observada implica causalidad.

### Contenido
1. Configuración del entorno
2. Librerías y versiones del entorno
3. Carga de la base y variables de corte
4. Funciones de estadísticos ponderados
5. Subconjuntos y ponderadores
6. Comparación de ponderadores para el ingreso medio
7. Ingreso de la ocupación principal por año
8. Ingreso laboral ampliado
9. Ingreso por edad, nivel educativo y región
10. Empleo, desempleo e ingreso cero
11. Resumen por etapa
12. Correlación entre educación e ingreso
13. Sensibilidad al deflactor
14. Resumen y limitaciones

## 1. Configuración del entorno
Se monta Google Drive y se definen las rutas del proyecto. El control final confirma que existen los dos archivos de entrada y la carpeta de resultados.

In [1]:
from google.colab import drive   # herramienta de Colab para conectar Drive

# Colab pide autorización la primera vez en cada sesión
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from pathlib import Path   # rutas independientes del sistema operativo

# Rutas del proyecto (las mismas constantes que en 01_preparacion)
RAIZ             = Path('/content/drive/MyDrive/eph_ingresos')
DATOS_PROCESADOS = RAIZ / 'datos_procesados'
DATOS_IPC        = RAIZ / 'datos_originales' / 'ipc'
RESULTADOS       = RAIZ / 'resultados'

# Archivos de entrada
ARCHIVO_ENTRADA = DATOS_PROCESADOS / 'eph_t2_2017_2026_18mas_ingresos.parquet'
ARCHIVO_IPC     = DATOS_IPC / 'serie_ipc_divisiones.csv'   # solo para la sección 13

# Control: todo debe dar True
for ruta in [ARCHIVO_ENTRADA, ARCHIVO_IPC, RESULTADOS]:
    print(f'{ruta.name:45s}', ruta.exists())

eph_t2_2017_2026_18mas_ingresos.parquet       True
serie_ipc_divisiones.csv                      True
resultados                                    True


## 2. Librerías y versiones del entorno
Se importan todas las librerías del notebook y se muestran sus versiones. Las versiones de referencia del proyecto son Python 3.13.18, pandas 2.2.3 y numpy 2.4.6. Si una ejecución futura muestra otras y los resultados cambian, se comparan con estas.

In [3]:
import sys
import numpy as np                   # cálculos numéricos
import pandas as pd                  # manejo de tablas
import scipy
from scipy.stats import spearmanr    # solo para controlar la Spearman propia (sección 12)

print('Python:', sys.version.split()[0])   # split()[0] deja solo el número de versión
print('pandas:', pd.__version__)
print('numpy :', np.__version__)
print('scipy :', scipy.__version__)

Python: 3.13.18
pandas: 2.2.3
numpy : 2.4.6
scipy : 1.16.3


## 3. Carga de la base y variables de corte

### 3.1 Carga de la base
Se lee la base generada por `01_preparacion` y se comprueba que tenga las dimensiones registradas al cierre de la Etapa 3.

In [4]:
eph = pd.read_parquet(ARCHIVO_ENTRADA)

# Control: dimensiones registradas al cierre de la Etapa 3
print('Forma:', eph.shape)
assert eph.shape == (369_946, 20), 'La base no coincide con la salida de 01_preparacion'

# Nombres, tipos y valores no nulos de cada columna
eph.info()

Forma: (369946, 20)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 369946 entries, 0 to 369945
Data columns (total 20 columns):
 #   Column           Non-Null Count   Dtype   
---  ------           --------------   -----   
 0   ANO4             369946 non-null  int64   
 1   CODUSU           369946 non-null  object  
 2   NRO_HOGAR        369946 non-null  int64   
 3   COMPONENTE       369946 non-null  int64   
 4   AGLOMERADO       369946 non-null  int64   
 5   REGION           369946 non-null  int64   
 6   CH06             369946 non-null  int64   
 7   NIVEL_ED         369946 non-null  int64   
 8   ESTADO           369946 non-null  int64   
 9   P21              369946 non-null  int64   
 10  TOT_P12          369946 non-null  int64   
 11  PONDIIO          369946 non-null  int64   
 12  PONDERA          369946 non-null  int64   
 13  grupo_edad       369946 non-null  category
 14  nivel_educativo  369946 non-null  object  
 15  etapa            369946 non-null  category
 16  

La base tiene 369.946 filas y 20 columnas. Las únicas columnas con valores faltantes son `p21_real` (171.253 casos válidos) e `ing_lab_real` (170.087): los faltantes corresponden a los casos sin un ingreso válido, por diseño de la Etapa 3.

### 3.2 Orden y nombres de las variables de corte
El nivel educativo se convierte en una categoría ordenada de menor a mayor, y los códigos de región se reemplazan por sus nombres. Así, las tablas y los gráficos respetan siempre el mismo orden.

In [5]:
# Nivel educativo: categoría ordenada de menor a mayor nivel
ORDEN_EDUC = [
    'Primario incompleto', 'Primario completo',
    'Secundario incompleto', 'Secundario completo',
    'Superior incompleto', 'Superior completo',
]
eph['nivel_educativo'] = pd.Categorical(
    eph['nivel_educativo'], categories=ORDEN_EDUC, ordered=True
)

# Región: códigos de la EPH -> nombres, en el orden de la planificación
NOMBRES_REGION = {
    1: 'Gran Buenos Aires', 40: 'Noroeste', 41: 'Noreste',
    42: 'Cuyo', 43: 'Pampeana', 44: 'Patagonia',
}
eph['region'] = pd.Categorical(
    eph['REGION'].map(NOMBRES_REGION),          # reemplaza cada código por su nombre
    categories=list(NOMBRES_REGION.values()),   # fija el orden de las categorías
    ordered=True,
)

# Control: ningún valor debe quedar sin categoría
assert eph['nivel_educativo'].notna().all() and eph['region'].notna().all()
print(eph['region'].value_counts(sort=False))   # casos sin ponderar, mismo orden que NOMBRES_REGION

region
Gran Buenos Aires     57292
Noroeste              78876
Noreste               37409
Cuyo                  38223
Pampeana             113107
Patagonia             45039
Name: count, dtype: int64


## 4. Funciones de estadísticos ponderados
pandas no ofrece una mediana ponderada, así que se definen tres funciones propias:

$$\bar{x}_w = \frac{\sum_i w_i\,x_i}{\sum_i w_i} \qquad\qquad s_w = \sqrt{\frac{\sum_i w_i\,(x_i - \bar{x}_w)^2}{\sum_i w_i}}$$

El percentil ponderado ordena los ingresos, acumula los pesos y devuelve el primer ingreso cuyo peso acumulado alcanza la proporción $q$ del peso total. La celda termina con una prueba sobre un ejemplo que se puede resolver a mano. En la sección 5.3 se controlan con datos reales.

In [6]:
def media_ponderada(valores, pesos):
    """Media ponderada: suma(x * w) / suma(w)."""
    return np.sum(valores * pesos) / np.sum(pesos)


def percentil_ponderado(valores, pesos, q):
    """Percentil ponderado (q entre 0 y 1): primer valor cuyo peso acumulado alcanza q."""
    v = np.asarray(valores)
    w = np.asarray(pesos)
    orden = np.argsort(v)                    # posiciones que ordenan los ingresos
    v, w = v[orden], w[orden]                # valores y pesos con el mismo orden
    acumulado = np.cumsum(w)                 # se acumulan los pesos, no los ingresos
    posicion = np.searchsorted(acumulado, q * w.sum())
    return v[posicion]


def desvio_ponderado(valores, pesos):
    """Desvío estándar ponderado: raíz de suma(w * (x - media)**2) / suma(w)."""
    media = media_ponderada(valores, pesos)
    varianza = np.sum(pesos * (valores - media)**2) / np.sum(pesos)
    return np.sqrt(varianza)


# Prueba con un ejemplo que se puede resolver a mano
x = pd.Series([100, 200, 300])
w = pd.Series([10, 10, 80])
print('Media ponderada  :', media_ponderada(x, w))           # esperado: 270.0
print('Mediana ponderada:', percentil_ponderado(x, w, 0.5))  # esperado: 300
print('Desvío ponderado :', round(desvio_ponderado(x, w), 2))  # esperado: 64.03

Media ponderada  : 270.0
Mediana ponderada: 300
Desvío ponderado : 64.03


## 5. Subconjuntos y ponderadores
Cada indicador tiene su universo de casos y su ponderador:

| Subconjunto | Casos | Ponderador |
|---|---|---|
| `base_p21` | Ocupados con P21 > 0 | PONDIIO |
| `base_amp` | Ocupados con P21 > 0 y TOT_P12 ≥ 0 | PONDIIO |
| `base_tasas` | Personas de 18 años o más con ESTADO ≠ 0 | PONDERA |
| `ocupados` | ESTADO = 1 (proporción con ingreso cero) | PONDERA |

In [7]:
# Ingreso de la ocupación principal: p21_real ya tiene NaN en los casos no válidos
base_p21 = eph[eph['p21_real'].notna()]

# Ingreso laboral ampliado: ing_lab_real ya tiene NaN si P21 <= 0 o TOT_P12 = -9
base_amp = eph[eph['ing_lab_real'].notna()]

# Tasas: se excluye ESTADO = 0 (entrevista individual no realizada)
base_tasas = eph[eph['ESTADO'].ne(0)]

# Ocupados: denominador de la proporción con ingreso cero
ocupados = eph[eph['ESTADO'].eq(1)]

tamanios = pd.Series({'base_p21': len(base_p21), 'base_amp': len(base_amp),
                      'base_tasas': len(base_tasas), 'ocupados': len(ocupados)})
print(tamanios)
assert tamanios.tolist() == [171_253, 170_087, 369_292, 208_828]

base_p21      171253
base_amp      170087
base_tasas    369292
ocupados      208828
dtype: int64


### 5.1 Controles de los ponderadores
Se comprueba que PONDIIO funcione como indica el INDEC: a) vale 0 en los ocupados que no respondieron el ingreso; b) es positivo en todos los casos válidos, y c) el peso total se conserva, es decir, el peso de quienes no respondieron se reparte entre quienes sí lo hicieron.

In [8]:
# a) Los ocupados que no respondieron P21 deben tener PONDIIO = 0
no_responde = ocupados[ocupados['P21'].eq(-9)]
print('Ocupados con P21 = -9:', len(no_responde))
print('a) PONDIIO = 0 en todos los que no respondieron:', no_responde['PONDIIO'].eq(0).all())

# b) Los casos válidos deben tener PONDIIO positivo
print('b) PONDIIO > 0 en base_p21 y base_amp        :',
      base_p21['PONDIIO'].gt(0).all() and base_amp['PONDIIO'].gt(0).all())

# c) El peso total se conserva: PONDIIO de quienes respondieron frente a PONDERA de todos
control_pesos = pd.DataFrame({
    'suma_PONDIIO_respondentes': ocupados[ocupados['P21'].ge(0)].groupby('ANO4')['PONDIIO'].sum(),
    'suma_PONDERA_ocupados': ocupados.groupby('ANO4')['PONDERA'].sum(),
})
control_pesos['cociente'] = (control_pesos['suma_PONDIIO_respondentes']
                             / control_pesos['suma_PONDERA_ocupados']).round(4)
control_pesos

Ocupados con P21 = -9: 32273
a) PONDIIO = 0 en todos los que no respondieron: True
b) PONDIIO > 0 en base_p21 y base_amp        : True


,suma_PONDIIO_respondentes,suma_PONDERA_ocupados,cociente
ANO4,,,
2017,11339856,11339502,1.0000
2018,11590838,11593387,0.9998
2019,12002119,11999537,1.0002
2020,9508189,9510010,0.9998
2021,11882022,11904049,0.9981
2022,12928484,12932025,0.9997
2023,13081020,13080482,1.0000
2024,13224184,13231013,0.9995
2025,13257566,13260961,0.9997


Los tres controles se cumplen. Los 32.273 ocupados con P21 = −9 tienen PONDIIO = 0, y el cociente entre la suma de PONDIIO de quienes respondieron y la suma de PONDERA de todos los ocupados está entre 0,9981 y 1,0002 en todos los años. PONDIIO no deja afuera a la población que no respondió: redistribuye su peso.

### 5.2 No respuesta del ingreso, con y sin ponderar
La no respuesta de P21 se mide de dos formas: como proporción de los casos de la muestra y como proporción de la población representada (con PONDERA). La segunda tabla muestra la no respuesta ponderada por región.

In [9]:
# Proporción de ocupados con P21 = -9: casos de la muestra y población representada (PONDERA)
no_respuesta = ocupados.groupby('ANO4').apply(lambda g: pd.Series({
    'sin_ponderar_%': g['P21'].eq(-9).mean() * 100,
    'ponderada_%': media_ponderada(g['P21'].eq(-9), g['PONDERA']) * 100,
}), include_groups=False)
display(no_respuesta.round(1))

# Ponderada, por región: dónde se concentra la no respuesta
no_respuesta_region = (ocupados.groupby(['ANO4', 'region'], observed=True)
                               .apply(lambda g: media_ponderada(g['P21'].eq(-9), g['PONDERA']) * 100,
                                      include_groups=False)
                               .unstack())
display(no_respuesta_region.round(1))

,sin_ponderar_%,ponderada_%
ANO4,,
2017,16.1,20.8
2018,14.5,20.7
2019,10.8,18.0
2020,11.3,18.2
2021,14.1,20.4
2022,16.0,23.6
2023,17.0,24.2
2024,17.5,22.9
2025,18.0,21.9


region,Gran Buenos Aires,Noroeste,Noreste,Cuyo,Pampeana,Patagonia
ANO4,,,,,,
2017,24.1,6.4,13.0,7.7,24.8,13.4
2018,27.4,4.1,17.6,7.1,17.0,10.3
2019,24.8,2.1,5.5,5.2,15.7,5.8
2020,23.4,2.4,5.8,8.8,19.8,10.4
2021,24.5,2.2,7.7,10.4,25.2,11.9
2022,31.2,3.6,9.4,8.9,22.5,13.3
2023,30.6,4.8,15.4,11.2,24.9,11.3
2024,28.2,4.8,13.9,15.1,22.7,16.0
2025,26.2,4.1,24.4,20.5,19.8,17.6


- **La no respuesta ponderada es más alta que la de la muestra:** va del 20,8 % en 2017 al 25,3 % en 2026, frente al 16,1 % y el 18,5 % sin ponderar.
- **Se concentra en Gran Buenos Aires** (entre el 23,4 % y el 32,0 %), que representa alrededor del 54 % de los ocupados, y en la región Pampeana. En el Noroeste no supera el 6,4 %.
- **En Noreste, Cuyo y Patagonia aumenta en los últimos años:** en 2025 y 2026 supera el 16 % en las tres regiones.

PONDIIO corrige la no respuesta bajo el supuesto de que, dentro de cada grupo de ajuste, quienes no responden se parecen a quienes sí lo hacen. Con una cuarta parte de la población sin ingreso declarado, ese supuesto pesa más en los últimos años.

### 5.3 Control de las funciones con datos reales
Las funciones propias se comparan con numpy sobre el indicador central de 2017. Para los percentiles se usa `method='inverted_cdf'`, que aplica el mismo criterio sin interpolación.

In [10]:
# Control de las funciones propias contra numpy con datos reales (indicador central, 2017)
m = base_p21[base_p21['ANO4'].eq(2017)]
x, w = m['p21_real'], m['PONDIIO']

print('Media     – propia:', round(media_ponderada(x, w), 4), ' numpy:', round(np.average(x, weights=w), 4))
for q in (0.10, 0.25, 0.50, 0.75, 0.90):
    propio = percentil_ponderado(x, w, q)
    ref = np.quantile(x, q, weights=w, method='inverted_cdf')   # mismo criterio, sin interpolar
    print(f'p{int(q * 100):<3d}     – propia: {propio:,.2f}  numpy: {ref:,.2f}  iguales: {propio == ref}')

Media     – propia: 1521537.4717  numpy: 1521537.4717
p10      – propia: 353,371.31  numpy: 353,371.31  iguales: True
p25      – propia: 706,742.62  numpy: 706,742.62  iguales: True
p50      – propia: 1,177,904.37  numpy: 1,177,904.37  iguales: True
p75      – propia: 1,884,647.00  numpy: 1,884,647.00  iguales: True
p90      – propia: 2,944,760.93  numpy: 2,944,760.93  iguales: True


La media y los cinco percentiles coinciden exactamente con numpy.

## 6. Comparación de ponderadores para el ingreso medio
Esta sección se conserva como evidencia de la elección de PONDIIO. Se calcula la media del indicador central con los dos ponderadores y su diferencia relativa.

In [11]:
def resumen_ingreso(g, col='p21_real'):
    """Casos, media con cada ponderador y mediana con PONDIIO."""
    return pd.Series({
        'n': len(g),                                                  # casos sin ponderar
        'media_pondiio': media_ponderada(g[col], g['PONDIIO']),       # ponderador elegido
        'media_pondera': media_ponderada(g[col], g['PONDERA']),       # solo para comparar
        'mediana': percentil_ponderado(g[col], g['PONDIIO'], 0.5),
    })

# include_groups=False evita un aviso de pandas 2.2: ANO4 no se pasa a la función
comparacion_ponderadores = base_p21.groupby('ANO4').apply(resumen_ingreso, include_groups=False)
comparacion_ponderadores['n'] = comparacion_ponderadores['n'].astype('int64')

# Diferencia relativa de la media con PONDERA respecto de PONDIIO, en porcentaje
comparacion_ponderadores['dif_%'] = (comparacion_ponderadores['media_pondera']
                                     / comparacion_ponderadores['media_pondiio'] - 1) * 100
comparacion_ponderadores.round(1)

,n,media_pondiio,media_pondera,mediana,dif_%
ANO4,,,,,
2017,19632,1521537.5,1512335.1,1177904.4,-0.6
2018,19517,1520488.0,1500312.0,1311550.9,-1.3
2019,20985,1319634.0,1302820.2,1141283.0,-1.3
2020,10625,1317221.6,1302069.7,1112914.0,-1.2
2021,16072,1273113.9,1249203.2,1002548.0,-1.9
2022,18058,1192885.9,1176581.0,981221.9,-1.4
2023,17590,1209816.7,1201255.4,1024704.4,-0.7
2024,16659,1057478.7,1036140.0,822220.9,-2.0
2025,16220,1377898.1,1354983.4,1046662.3,-1.7


Con PONDERA, la media da entre un 0,5 % y un 2,0 % menos todos los años. PONDIIO da más peso a los grupos con más no respuesta y, como la media sube, esos grupos tienen ingresos más altos. Usar PONDERA subestimaría el ingreso medio.

## 7. Ingreso de la ocupación principal por año
Se calcula el resumen completo del indicador central para cada año: casos, media, mediana, desvío, percentiles 10, 25, 75 y 90, rango intercuartílico (RIQ) y cociente p90/p10.

In [12]:
def resumen_completo(g, col='p21_real', peso='PONDIIO'):
    """Casos, media, percentiles y medidas de dispersión de un grupo."""
    x, w = g[col], g[peso]
    p10, p25, p50, p75, p90 = [percentil_ponderado(x, w, q)
                               for q in (0.10, 0.25, 0.50, 0.75, 0.90)]
    return pd.Series({
        'n': len(g),
        'media': media_ponderada(x, w),
        'mediana': p50,
        'desvio': desvio_ponderado(x, w),
        'p10': p10, 'p25': p25, 'p75': p75, 'p90': p90,
        'riq': p75 - p25,          # rango intercuartílico (dispersión principal)
        'p90_p10': p90 / p10,      # cociente entre percentiles extremos
    })


p21_anual = base_p21.groupby('ANO4').apply(resumen_completo, include_groups=False)
p21_anual['n'] = p21_anual['n'].astype('int64')   # la cantidad de casos se guarda como entero

p21_anual.to_csv(RESULTADOS / 'p21_real_anual.csv')
print('Guardado:', (RESULTADOS / 'p21_real_anual.csv').exists())
p21_anual.round(2)

Guardado: True


,n,media,mediana,desvio,p10,p25,p75,p90,riq,p90_p10
ANO4,,,,,,,,,,
2017,19632,1521537.47,1177904.37,1313912.47,353371.31,706742.62,1884647.00,2944760.93,1177904.37,8.33
2018,19517,1520488.04,1311550.92,1245542.77,374728.84,749457.67,1873644.18,2810466.26,1124186.51,7.50
2019,20985,1319633.99,1141283.04,1163275.53,300337.64,600675.29,1741958.33,2402701.15,1141283.04,8.00
2020,10625,1317221.61,1112913.98,1108313.70,309142.77,618285.54,1648761.45,2473142.18,1030475.91,8.00
2021,16072,1273113.93,1002547.98,1255202.89,282407.88,564815.76,1637965.71,2259263.05,1073149.95,8.00
2022,18058,1192885.88,981221.91,1008793.43,285966.31,536186.83,1501323.14,2323476.28,965136.30,8.12
2023,17590,1209816.74,1024704.39,1117521.59,256176.10,512352.20,1596831.01,2305584.89,1084478.82,9.00
2024,16659,1057478.72,822220.86,1072006.45,177777.48,444443.71,1333331.13,1999996.69,888887.42,11.25
2025,16220,1377898.13,1046662.28,1631306.30,299046.37,598092.73,1644755.02,2691417.30,1046662.28,9.00


- **Media:** baja de 1.521.537 pesos en 2017 a 1.276.792 en 2026 (−16,1 %).
- **Mediana:** es máxima en 2018 (1.311.551), mínima en 2024 (822.221) y llega a 1.013.936 en 2026 (−13,9 % frente a 2017).
- **2024 es el mínimo de la serie** con todas las medidas de posición; en 2025 hay un repunte (+30,3 % en la media).
- **Dispersión:** el RIQ se mantiene entre 0,89 y 1,18 millones. El cociente p90/p10 toma valores simples, como 7,5; 8; 9 u 11,25, porque los ingresos se declaran en montos redondos. Por ese agrupamiento, la mediana se acompaña siempre con la media.
- **El desvío salta en 2025** (1,63 millones) mientras el RIQ se mantiene en niveles habituales: es sensible a la cola superior.

### 7.1 Valores extremos
Los ingresos extremos no se recortan. Para medir su efecto, se compara la media con la que resulta de excluir el 1 % de ingresos más altos.

In [13]:
def extremos(g, col='p21_real', peso='PONDIIO'):
    """Peso de la cola superior: máximo y media sin el 1 % más alto."""
    x, w = g[col], g[peso]
    p99 = percentil_ponderado(x, w, 0.99)
    debajo = x <= p99
    return pd.Series({
        'maximo': x.max(),
        'max_sobre_mediana': x.max() / percentil_ponderado(x, w, 0.5),
        'media': media_ponderada(x, w),
        'media_sin_top1': media_ponderada(x[debajo], w[debajo]),
    })

tabla_extremos = base_p21.groupby('ANO4').apply(extremos, include_groups=False)

# Variación interanual de la media, con y sin el 1 % más alto
tabla_extremos['var_media_%'] = tabla_extremos['media'].pct_change() * 100
tabla_extremos['var_sin_top1_%'] = tabla_extremos['media_sin_top1'].pct_change() * 100
display(tabla_extremos.round(1))

for col in ['media', 'media_sin_top1']:
    var = (tabla_extremos.loc[2026, col] / tabla_extremos.loc[2017, col] - 1) * 100
    print(f'Variación 2017–2026 de {col:15s}: {var:.1f} %')

,maximo,max_sobre_mediana,media,media_sin_top1,var_media_%,var_sin_top1_%
ANO4,,,,,,
2017,28269705.0,24.0,1521537.5,1459243.8,NaN,NaN
2018,32788773.1,25.0,1520488.0,1455773.9,-0.1,-0.2
2019,30033764.3,26.3,1319634.0,1251151.7,-13.2,-14.1
2020,22464374.8,20.2,1317221.6,1261920.0,-0.2,0.9
2021,28240759.9,28.2,1273113.9,1204770.6,-3.3,-4.5
2022,17872894.5,18.2,1192885.9,1139742.5,-6.3,-5.4
2023,21348008.2,20.8,1209816.7,1157983.4,1.4,1.6
2024,88888741.8,108.1,1057478.7,993878.4,-12.6,-14.2
2025,44856955.0,42.9,1377898.1,1300773.5,30.3,30.9


Variación 2017–2026 de media          : -16.1 %
Variación 2017–2026 de media_sin_top1 : -17.9 %


- **El máximo de 2024 equivale a 108 veces la mediana**; en los demás años, entre 18 y 43 veces.
- **Los extremos no cambian las conclusiones:** sin el 1 % más alto, la caída entre 2017 y 2026 es del −17,9 % (−16,1 % con todos los casos), la de 2024 del −14,2 % y el repunte de 2025 del +30,9 %.

## 8. Ingreso laboral ampliado
Se repite el resumen para el indicador ampliado (P21 + TOT_P12) y se compara con el central.

In [14]:
# Misma función, otra base y otra columna; el ponderador sigue siendo PONDIIO
amp_anual = base_amp.groupby('ANO4').apply(resumen_completo, col='ing_lab_real', include_groups=False)
amp_anual['n'] = amp_anual['n'].astype('int64')

amp_anual.to_csv(RESULTADOS / 'ing_lab_real_anual.csv')
print('Guardado:', (RESULTADOS / 'ing_lab_real_anual.csv').exists())

# Comparación entre los dos indicadores
comparacion = pd.DataFrame({
    'media_p21': p21_anual['media'],
    'media_amp': amp_anual['media'],
    'mediana_p21': p21_anual['mediana'],
    'mediana_amp': amp_anual['mediana'],
})
comparacion['dif_media_%'] = (comparacion['media_amp'] / comparacion['media_p21'] - 1) * 100
comparacion['dif_mediana_%'] = (comparacion['mediana_amp'] / comparacion['mediana_p21'] - 1) * 100
comparacion.round(1)

Guardado: True


,media_p21,media_amp,mediana_p21,mediana_amp,dif_media_%,dif_mediana_%
ANO4,,,,,,
2017,1521537.5,1590908.9,1177904.4,1295694.8,4.6,10.0
2018,1520488.0,1592406.6,1311550.9,1405233.1,4.7,7.1
2019,1319634.0,1389360.4,1141283.0,1201350.6,5.3,5.3
2020,1317221.6,1367240.7,1112914.0,1154133.0,3.8,3.7
2021,1273113.9,1345324.4,1002548.0,1073149.9,5.7,7.0
2022,1192885.9,1258541.2,981221.9,1072373.7,5.5,9.3
2023,1209816.7,1274515.3,1024704.4,1024704.4,5.3,0.0
2024,1057478.7,1114929.9,822220.9,888887.4,5.4,8.1
2025,1377898.1,1458790.0,1046662.3,1196185.5,5.9,14.3


- La media del indicador ampliado supera a la del central entre un 3,8 % (2020) y un 5,9 % (2025). Sigue la misma evolución.
- La diferencia entre medianas oscila entre 0 % (2023) y 14,3 % (2025): con montos redondos, la mediana salta de un valor declarado a otro. Por eso la comparación entre indicadores se hace con la media.

## 9. Ingreso por edad, nivel educativo y región

### 9.1 Casos por celda
Antes de comparar grupos, se cuentan los casos sin ponderar de cada cruce año × grupo, para aplicar el criterio de precisión.

In [15]:
CORTES = {'edad': 'grupo_edad', 'educacion': 'nivel_educativo', 'region': 'region'}

# Casos sin ponderar de p21_real en cada cruce año × grupo
minimos = {}
for nombre, var in CORTES.items():
    conteo = pd.crosstab(base_p21['ANO4'], base_p21[var])
    print(f'--- Casos de p21_real: año × {nombre} ---')
    display(conteo)
    minimos[nombre] = conteo.min().min()

print('Celda con menos casos en cada corte:', minimos)

--- Casos de p21_real: año × edad ---


grupo_edad,18-29,30-39,40-64,65 o más
ANO4,,,,
2017,4471,5247,9241,673
2018,4452,5211,9188,666
2019,4631,5571,10007,776
2020,2078,2800,5439,308
2021,3468,4191,7902,511
2022,4025,4764,8618,651
2023,3938,4651,8460,541
2024,3610,4359,8083,607
2025,3312,4277,8019,612


--- Casos de p21_real: año × educacion ---


nivel_educativo,Primario incompleto,Primario completo,Secundario incompleto,Secundario completo,Superior incompleto,Superior completo
ANO4,,,,,,
2017,747,3003,3350,5715,2685,4132
2018,744,2835,3303,5812,2617,4206
2019,839,2904,3499,6192,2848,4703
2020,310,1316,1659,3170,1488,2682
2021,559,1880,2713,4919,2298,3703
2022,656,2117,3062,5417,2660,4146
2023,565,1982,3013,5507,2581,3942
2024,503,1870,2770,5285,2346,3885
2025,435,1660,2554,5282,2449,3840


--- Casos de p21_real: año × region ---


region,Gran Buenos Aires,Noroeste,Noreste,Cuyo,Pampeana,Patagonia
ANO4,,,,,,
2017,3490,4475,1832,2093,5246,2496
2018,3153,4492,1801,2031,5523,2517
2019,3167,4752,2105,2162,6088,2711
2020,994,2770,1176,1614,2956,1115
2021,1169,4497,1742,1953,4448,2263
2022,2311,4523,1899,2027,4952,2346
2023,2261,4247,1808,2031,4899,2344
2024,2352,4010,1682,1761,4751,2103
2025,2328,4131,1417,1708,4721,1915


Celda con menos casos en cada corte: {'edad': 308, 'educacion': 310, 'region': 994}


La celda más chica tiene 308 casos (65 años o más en 2020). Todas superan los 100 casos, así que los 160 cruces son publicables. Las celdas más débiles son las de 65 años o más, primario incompleto y Gran Buenos Aires en 2020 y 2021.

### 9.2 Estadísticos por grupo
Se calcula el resumen completo para cada año y grupo, se clasifica la precisión de cada celda y se guardan las tres tablas.

In [16]:
def tabla_por_corte(base, var, col='p21_real'):
    """Resumen completo por año y grupo, con la clasificación de precisión."""
    tabla = (base.groupby(['ANO4', var], observed=True)
                 .apply(resumen_completo, col=col, include_groups=False))
    tabla['n'] = tabla['n'].astype('int64')
    # Criterio de precisión: hasta 29 casos no se publica, de 30 a 99 con advertencia
    tabla['precision'] = pd.cut(tabla['n'], bins=[0, 29, 99, np.inf],
                                labels=['no publicar', 'advertencia', 'publicable'])
    return tabla

tablas = {}
for nombre, var in CORTES.items():
    tablas[nombre] = tabla_por_corte(base_p21, var)
    archivo = RESULTADOS / f'p21_real_anual_{nombre}.csv'
    tablas[nombre].to_csv(archivo)
    print(f'{nombre:10s} celdas: {len(tablas[nombre]):3d}  guardado: {archivo.exists()}  ',
          tablas[nombre]['precision'].value_counts().to_dict())

edad       celdas:  40  guardado: True   {'publicable': 40, 'no publicar': 0, 'advertencia': 0}
educacion  celdas:  60  guardado: True   {'publicable': 60, 'no publicar': 0, 'advertencia': 0}
region     celdas:  60  guardado: True   {'publicable': 60, 'no publicar': 0, 'advertencia': 0}


### 9.3 Media por grupo y variación 2017–2026
Se muestra la media de cada grupo por año y la variación entre 2017 y 2026 con la media y con la mediana.

In [17]:
for nombre in CORTES:
    print(f'\n--- Media de p21_real por año y {nombre} ---')
    display(tablas[nombre]['media'].unstack().round(0))

    # Variación % entre 2017 y 2026, para la media y la mediana
    t = tablas[nombre][['media', 'mediana']].unstack()   # columnas: (estadístico, grupo)
    variacion = (t.loc[2026] / t.loc[2017] - 1) * 100     # una fila por (estadístico, grupo)
    print(f'--- Variación % 2017–2026 por {nombre} ---')
    display(variacion.unstack(level=0).round(1))          # filas: grupo; columnas: media y mediana


--- Media de p21_real por año y edad ---


grupo_edad,18-29,30-39,40-64,65 o más
ANO4,,,,
2017,1139030.0,1558850.0,1697892.0,1532306.0
2018,1143791.0,1560021.0,1684848.0,1553880.0
2019,963630.0,1314316.0,1510796.0,1099567.0
2020,901726.0,1332886.0,1477227.0,1244431.0
2021,860290.0,1255873.0,1441452.0,1523467.0
2022,870462.0,1261600.0,1313365.0,1160981.0
2023,899023.0,1240023.0,1351411.0,987483.0
2024,780698.0,1103548.0,1169686.0,859735.0
2025,939873.0,1393255.0,1568157.0,1273946.0


--- Variación % 2017–2026 por edad ---


,media,mediana
grupo_edad,,
18-29,-18.4,-18.4
30-39,-16.6,-20.3
40-64,-17.8,-20.3
65 o más,-6.2,-28.3



--- Media de p21_real por año y educacion ---


nivel_educativo,Primario incompleto,Primario completo,Secundario incompleto,Secundario completo,Superior incompleto,Superior completo
ANO4,,,,,,
2017,894439.0,1087378.0,1220405.0,1442812.0,1537493.0,2291786.0
2018,872518.0,1101662.0,1158522.0,1443753.0,1484582.0,2303092.0
2019,735901.0,943120.0,971453.0,1195416.0,1359882.0,1986135.0
2020,692947.0,914629.0,928356.0,1178538.0,1275968.0,1964159.0
2021,686988.0,834944.0,911805.0,1143114.0,1179018.0,1946213.0
2022,703526.0,794808.0,843389.0,1083926.0,1297274.0,1756868.0
2023,605191.0,874853.0,838892.0,1114316.0,1200057.0,1835116.0
2024,570074.0,615435.0,776433.0,950956.0,1130004.0,1563578.0
2025,760702.0,862295.0,980441.0,1240197.0,1423022.0,2076907.0


... (2 outputs left out)


--- Media de p21_real por año y region ---


region,Gran Buenos Aires,Noroeste,Noreste,Cuyo,Pampeana,Patagonia
ANO4,,,,,,
2017,1613143.0,1155562.0,1200105.0,1320325.0,1464148.0,2141353.0
2018,1594758.0,1193556.0,1236203.0,1310572.0,1485022.0,2111868.0
2019,1396644.0,997004.0,1109721.0,1097790.0,1285702.0,1793911.0
2020,1428175.0,1039411.0,976450.0,1039266.0,1256971.0,1738115.0
2021,1409236.0,872015.0,924143.0,1001823.0,1193830.0,1728186.0
2022,1263182.0,885548.0,927475.0,993578.0,1182066.0,1679539.0
2023,1300605.0,872501.0,951788.0,994202.0,1165395.0,1700298.0
2024,1156680.0,736141.0,669367.0,851387.0,1030898.0,1390787.0
2025,1525725.0,959846.0,882207.0,1089813.0,1303979.0,1815580.0


--- Variación % 2017–2026 por region ---


,media,mediana
region,,
Gran Buenos Aires,-14.1,-20.3
Noroeste,-16.8,-16.3
Noreste,-27.6,-25.6
Cuyo,-22.2,-23.5
Pampeana,-16.5,-18.0
Patagonia,-18.1,-28.3


- **Edad:** los grupos de 18 a 64 años caen entre −16,6 % y −18,4 % con la media. El grupo de 18 a 29 años tiene siempre el ingreso más bajo. Con la media, el grupo de 40 a 64 años supera al de 30 a 39 entre un 4,1 % y un 14,9 %. El grupo de 65 años o más es el menos preciso y no es concluyente.
- **Educación:** el ingreso crece con el nivel educativo en todos los años, con una sola excepción: en 2023, secundario incompleto queda apenas por debajo de primario completo. Superior completo es el grupo que menos cae (−17,0 %). La brecha de medias entre superior completo y primario incompleto pasa de 2,56 a 2,83 veces.
- **Región:** Noreste es la que más cae (−27,6 % con la media y −25,6 % con la mediana). Patagonia tiene siempre el ingreso más alto. La brecha de medias entre Patagonia y Noroeste se mantiene (1,85 en 2017 y 1,82 en 2026).

### 9.4 Robustez de las variaciones
Se aplica el criterio de robustez a la variación 2017–2026 del total y de cada grupo: la media y la mediana deben coincidir en la dirección y diferir en menos de 4 puntos.

In [18]:
UMBRAL_PUNTOS = 4   # diferencia máxima entre media y mediana para considerar robusta una variación

def evaluar_robustez(media_ini, media_fin, mediana_ini, mediana_fin):
    """Variación % con media y mediana, y si cumple el criterio de robustez."""
    var_media = (media_fin / media_ini - 1) * 100
    var_mediana = (mediana_fin / mediana_ini - 1) * 100
    misma_direccion = np.sign(var_media) == np.sign(var_mediana)
    diferencia = abs(var_media - var_mediana)
    return var_media, var_mediana, diferencia, misma_direccion and diferencia < UMBRAL_PUNTOS

filas = []
# Total de adultos ocupados con ingreso
filas.append(('total', 'total', *evaluar_robustez(
    p21_anual.loc[2017, 'media'], p21_anual.loc[2026, 'media'],
    p21_anual.loc[2017, 'mediana'], p21_anual.loc[2026, 'mediana'])))
# Cada grupo de cada corte
for nombre in CORTES:
    t = tablas[nombre]
    for grupo in t.loc[2017].index:
        filas.append((nombre, grupo, *evaluar_robustez(
            t.loc[(2017, grupo), 'media'], t.loc[(2026, grupo), 'media'],
            t.loc[(2017, grupo), 'mediana'], t.loc[(2026, grupo), 'mediana'])))

robustez = pd.DataFrame(filas, columns=['corte', 'grupo', 'var_media_%', 'var_mediana_%',
                                        'diferencia_puntos', 'robusta'])
robustez.to_csv(RESULTADOS / 'robustez_variaciones.csv', index=False)
print('Guardado:', (RESULTADOS / 'robustez_variaciones.csv').exists())
robustez.round(1)

Guardado: True


,corte,grupo,var_media_%,var_mediana_%,diferencia_puntos,robusta
0,total,total,-16.1,-13.9,2.2,True
1,edad,18-29,-18.4,-18.4,0.1,True
2,edad,30-39,-16.6,-20.3,3.7,True
3,edad,40-64,-17.8,-20.3,2.5,True
4,edad,65 o más,-6.2,-28.3,22.0,False
5,educacion,Primario incompleto,-24.9,-36.2,11.4,False
6,educacion,Primario completo,-20.1,-28.3,8.1,False
7,educacion,Secundario incompleto,-26.5,-25.6,0.9,True
8,educacion,Secundario completo,-20.2,-21.7,1.6,True
9,educacion,Superior incompleto,-18.2,-28.3,10.1,False


- **Son robustas** la caída del total, la de los tres grupos de 18 a 64 años, la de secundario incompleto, secundario completo y superior completo, y la de Noroeste, Noreste, Cuyo y Pampeana.
- **No son robustas** las de 65 años o más, primario incompleto y completo, superior incompleto, Gran Buenos Aires y Patagonia. En todos esos casos la mediana cae mucho más que la media porque salta entre montos redondos. Se informan con la media.

## 10. Empleo, desempleo e ingreso cero
Se calculan con PONDERA la tasa de empleo (ocupados sobre la población de 18 años o más), la de desempleo (desocupados sobre la población económicamente activa) y la proporción de ocupados con ingreso cero. La tasa de inactividad se muestra como apoyo para leer 2020.

In [19]:
def resumen_tasas(g):
    """Tasas de empleo y desempleo de un grupo, en porcentaje."""
    pea = g[g['ESTADO'].isin([1, 2])]          # población económicamente activa
    return pd.Series({
        'n': len(g),
        'n_pea': len(pea),
        'tasa_empleo': media_ponderada(g['ESTADO'].eq(1), g['PONDERA']) * 100,
        'tasa_desempleo': media_ponderada(pea['ESTADO'].eq(2), pea['PONDERA']) * 100,
    })

tasas_anual = base_tasas.groupby('ANO4').apply(resumen_tasas, include_groups=False)
tasas_anual[['n', 'n_pea']] = tasas_anual[['n', 'n_pea']].astype('int64')

# Proporción de ocupados con ingreso cero, en porcentaje (PONDERA sobre todos los ocupados)
tasas_anual['ingreso_cero'] = ocupados.groupby('ANO4').apply(
    lambda g: media_ponderada(g['P21'].eq(0), g['PONDERA']) * 100, include_groups=False)

# Tasa de inactividad: ayuda a leer la caída del empleo en 2020
tasa_inactividad = base_tasas.groupby('ANO4').apply(
    lambda g: media_ponderada(g['ESTADO'].eq(3), g['PONDERA']) * 100, include_groups=False)

tasas_anual.to_csv(RESULTADOS / 'tasas_anual.csv')
print('Guardado:', (RESULTADOS / 'tasas_anual.csv').exists())
display(tasas_anual.round(1))
print('Tasa de inactividad (%):', tasa_inactividad.round(1).to_dict())

Guardado: True


,n,n_pea,tasa_empleo,tasa_desempleo,ingreso_cero
ANO4,,,,,
2017,43164,25961,56.9,8.7,2.6
2018,41992,25522,57.1,9.5,3.0
2019,43094,26629,57.4,10.6,2.9
2020,27300,14327,46.9,13.1,4.5
2021,34935,21051,57.7,9.6,2.4
2022,37714,23464,61.4,6.8,2.5
2023,36445,23003,61.5,6.2,2.3
2024,35491,22128,61.3,7.5,2.2
2025,35096,21686,60.9,7.6,2.8


Tasa de inactividad (%): {2017: 37.7, 2018: 36.9, 2019: 35.7, 2020: 46.0, 2021: 36.1, 2022: 34.1, 2023: 34.5, 2024: 33.7, 2025: 34.1, 2026: 33.3}


- **2020:** el empleo cae 10,5 puntos (de 57,4 % a 46,9 %) y el desempleo sube 2,5 puntos (de 10,6 % a 13,1 %). La caída del empleo se corresponde con un aumento de la inactividad, que pasa de 35,7 % a 46,0 %, más que con un aumento del desempleo.
- **Desde 2022:** el empleo se ubica entre 60,9 % y 61,5 %, por encima de los niveles de 2017–2019, y el desempleo entre 6,2 % y 7,9 %.
- **Ingreso cero:** ronda el 2,2 %–3,0 % y sube al 4,5 % en 2020.

### 10.1 Validación con la tasa oficial del INDEC
La tasa de desempleo del proyecto se compara con la publicada por el INDEC para el total de los 31 aglomerados urbanos. La tasa oficial incluye a todas las edades y la del proyecto a las personas de 18 años o más, así que se esperan diferencias chicas.

In [20]:
# Tasa de desocupación oficial del INDEC, 2.º trimestre, total de los 31 aglomerados urbanos.
# Fuente: INDEC, "Mercado de trabajo. Tasas e indicadores socioeconómicos (EPH)", informe de cada 2.º trimestre.
DESEMPLEO_OFICIAL = {2017: 8.7, 2018: 9.6, 2019: 10.6, 2020: 13.1, 2021: 9.6,
                     2022: 6.9, 2023: 6.2, 2024: 7.6, 2025: 7.6, 2026: 7.9}

validacion = pd.DataFrame({
    'proyecto_18_o_mas': tasas_anual['tasa_desempleo'].round(1),
    'oficial_INDEC': pd.Series(DESEMPLEO_OFICIAL),
})
validacion['diferencia_puntos'] = (validacion['proyecto_18_o_mas'] - validacion['oficial_INDEC']).round(1)
display(validacion)
print('Diferencia máxima (puntos):', validacion['diferencia_puntos'].abs().max())

,proyecto_18_o_mas,oficial_INDEC,diferencia_puntos
2017,8.7,8.7,0.0
2018,9.5,9.6,-0.1
2019,10.6,10.6,0.0
2020,13.1,13.1,0.0
2021,9.6,9.6,0.0
2022,6.8,6.9,-0.1
2023,6.2,6.2,0.0
2024,7.5,7.6,-0.1
2025,7.6,7.6,0.0
2026,7.9,7.9,0.0


Diferencia máxima (puntos): 0.1


La diferencia máxima es de 0,1 puntos. El cálculo de las tasas reproduce las cifras oficiales. La tasa de empleo no se valida así porque el INDEC la calcula sobre la población total, incluidos los menores.

### 10.2 Ingreso cero según el ponderador
La proporción de ocupados con ingreso cero se calcula con tres criterios: sin ponderar, con PONDERA sobre todos los ocupados (criterio elegido) y con PONDIIO sobre quienes respondieron el ingreso.

In [21]:
# Proporción de ocupados con ingreso cero con tres criterios
ingreso_cero = ocupados.groupby('ANO4').apply(lambda g: pd.Series({
    'sin_ponderar_%': g['P21'].eq(0).mean() * 100,
    'PONDERA_todos_%': media_ponderada(g['P21'].eq(0), g['PONDERA']) * 100,        # criterio elegido
    'PONDIIO_respondentes_%': media_ponderada(g.loc[g['P21'].ge(0), 'P21'].eq(0),
                                              g.loc[g['P21'].ge(0), 'PONDIIO']) * 100,
}), include_groups=False)
display(ingreso_cero.round(2))

# ¿Por qué coinciden los dos ponderados? En los casos con P21 = 0, PONDIIO es igual a PONDERA
ceros = ocupados[ocupados['P21'].eq(0)]
print('PONDIIO = PONDERA en todos los casos con ingreso cero:', ceros['PONDIIO'].eq(ceros['PONDERA']).all())

,sin_ponderar_%,PONDERA_todos_%,PONDIIO_respondentes_%
ANO4,,,
2017,2.56,2.64,2.64
2018,2.72,2.99,2.99
2019,2.62,2.94,2.94
2020,5.50,4.49,4.49
2021,2.78,2.36,2.36
2022,2.39,2.50,2.50
2023,2.05,2.28,2.28
2024,1.89,2.15,2.15
2025,2.17,2.82,2.82


PONDIIO = PONDERA en todos los casos con ingreso cero: True


- **Los dos criterios ponderados dan exactamente lo mismo** en los diez años. En los casos con ingreso cero, PONDIIO es igual a PONDERA, y la suma de pesos de los dos denominadores coincide (sección 5.1). La elección del ponderador no cambia el resultado.
- **Sin ponderar, el ingreso cero de 2020 es del 5,5 %; ponderado, del 4,5 %.** Se informa la cifra ponderada.

## 11. Resumen por etapa
Se promedian los valores anuales de cada etapa: antes (2017–2019), durante (2020–2021) y después (2022–2026) de la cuarentena. Primero los indicadores generales y después el ingreso por grupo.

In [22]:
# Año -> etapa, tomado de la base (la columna etapa existe desde 01_preparacion)
etapa_por_anio = eph.groupby('ANO4')['etapa'].first()

# 11.1 Indicadores generales
anual = pd.concat([
    p21_anual[['media', 'mediana', 'riq']].add_prefix('p21_'),
    amp_anual[['media']].add_prefix('amp_'),
    tasas_anual[['tasa_empleo', 'tasa_desempleo', 'ingreso_cero']],
], axis=1)
anual['etapa'] = anual.index.map(etapa_por_anio).astype(eph['etapa'].dtype)   # conserva el orden

resumen_etapas = anual.groupby('etapa', observed=True).mean()
resumen_etapas['anios'] = anual.groupby('etapa', observed=True).size()
assert resumen_etapas['anios'].tolist() == [3, 2, 5]   # antes, durante y después
resumen_etapas.to_csv(RESULTADOS / 'resumen_etapas.csv')
display(resumen_etapas.round(1))

# 11.2 Ingreso por etapa y grupo
for nombre, var in CORTES.items():
    t = tablas[nombre][['media', 'mediana']].reset_index()     # ANO4 y var pasan a ser columnas
    t['etapa'] = t['ANO4'].map(etapa_por_anio).astype(eph['etapa'].dtype)
    res = t.groupby(['etapa', var], observed=True)[['media', 'mediana']].mean()
    res.to_csv(RESULTADOS / f'resumen_etapas_{nombre}.csv')
    print(f'\n--- Media de p21_real por etapa y {nombre} ---')
    display(res['media'].unstack().round(0))

,p21_media,p21_mediana,p21_riq,amp_media,tasa_empleo,tasa_desempleo,ingreso_cero,anios
etapa,,,,,,,,
antes,1453886.5,1210246.1,1147791.3,1524225.3,57.2,9.6,2.9,3
durante,1295167.8,1057731.0,1051812.9,1356282.5,52.3,11.4,3.4,2
después,1222974.3,977749.0,1022352.0,1291381.7,61.3,7.2,2.4,5



--- Media de p21_real por etapa y edad ---


grupo_edad,18-29,30-39,40-64,65 o más
etapa,,,,
antes,1082150.0,1477729.0,1631179.0,1395251.0
durante,881008.0,1294380.0,1459339.0,1383949.0
después,883870.0,1259755.0,1359759.0,1143793.0



--- Media de p21_real por etapa y educacion ---


nivel_educativo,Primario incompleto,Primario completo,Secundario incompleto,Secundario completo,Superior incompleto,Superior completo
etapa,,,,,,
antes,834286.0,1044053.0,1116794.0,1360661.0,1460652.0,2193671.0
durante,689968.0,874786.0,920080.0,1160826.0,1227493.0,1955186.0
después,662316.0,803183.0,867168.0,1108171.0,1261556.0,1827146.0



--- Media de p21_real por etapa y region ---


region,Gran Buenos Aires,Noroeste,Noreste,Cuyo,Pampeana,Patagonia
etapa,,,,,,
antes,1534848.0,1115374.0,1182009.0,1242895.0,1411624.0,2015711.0
durante,1418706.0,955713.0,950296.0,1020545.0,1225400.0,1733151.0
después,1326298.0,883062.0,860015.0,991161.0,1180983.0,1667990.0


- **Ingreso medio:** 1.453.887 pesos antes, 1.295.168 durante y 1.222.974 después. El ingreso siguió bajando después de la etapa "durante".
- **Mercado de trabajo:** el empleo pasa de 57,2 % a 52,3 % y a 61,3 %; el desempleo, de 9,6 % a 11,4 % y a 7,2 %.
- En la etapa "después", el empleo y el desempleo están mejor que antes de la cuarentena, pero el ingreso real es menor. Es una descripción de lo observado, no una relación causal.

## 12. Correlación entre educación e ingreso

### 12.1 Funciones y controles
La correlación de Spearman mide si dos variables crecen juntas, aunque no lo hagan en línea recta. La versión ponderada calcula la posición de cada persona en la población representada (rango ponderado) y aplica sobre esos rangos una correlación de Pearson ponderada:

$$r_w = \frac{\sum_i w_i (x_i - \bar{x}_w)(y_i - \bar{y}_w)}{\sqrt{\sum_i w_i (x_i - \bar{x}_w)^2 \, \sum_i w_i (y_i - \bar{y}_w)^2}}$$

Se controla de tres formas: a) con un ejemplo conocido; b) con pesos iguales a 1, contra SciPy, y c) con PONDIIO, contra SciPy aplicado a los datos expandidos, donde cada persona se repite tantas veces como indica su peso.

In [23]:
def rango_ponderado(valores, pesos):
    """Posición de cada valor en la población representada (entre 0 y 1), con empates promediados."""
    df = pd.DataFrame({'v': np.asarray(valores), 'w': np.asarray(pesos, dtype=float)})
    peso_por_valor = df.groupby('v')['w'].sum().sort_index()    # peso total de cada valor distinto
    peso_menores = peso_por_valor.cumsum() - peso_por_valor      # peso de los valores menores
    rango = (peso_menores + peso_por_valor / 2) / peso_por_valor.sum()
    return df['v'].map(rango).to_numpy()                         # rango de cada persona


def pearson_ponderada(x, y, w):
    """Correlación de Pearson ponderada."""
    x, y, w = (np.asarray(a, dtype=float) for a in (x, y, w))
    mx = np.sum(w * x) / np.sum(w)
    my = np.sum(w * y) / np.sum(w)
    covarianza = np.sum(w * (x - mx) * (y - my))
    return covarianza / np.sqrt(np.sum(w * (x - mx)**2) * np.sum(w * (y - my)**2))


def spearman_ponderada(x, y, w):
    """Spearman ponderada: Pearson ponderada aplicada a los rangos ponderados."""
    return pearson_ponderada(rango_ponderado(x, w), rango_ponderado(y, w), w)


# Control a) ejemplo de cinco personas con pesos iguales
x_ej, y_ej, w_ej = [1, 2, 3, 4, 5], [300, 400, 500, 600, 10_000], [1, 1, 1, 1, 1]
print('a) Pearson :', round(pearson_ponderada(x_ej, y_ej, w_ej), 3), '(esperado: 0.725)')
print('   Spearman:', round(spearman_ponderada(x_ej, y_ej, w_ej), 3), '(esperado: 1.0)')

# Control b) con pesos = 1 debe coincidir con SciPy en datos reales (2017)
m = base_p21[base_p21['ANO4'].eq(2017)]
educ = m['nivel_educativo'].cat.codes + 1
print('b) SciPy   :', round(spearmanr(educ, m['p21_real']).statistic, 6))
print('   Propia  :', round(spearman_ponderada(educ, m['p21_real'], np.ones(len(m))), 6))

# Control c) con PONDIIO debe coincidir con SciPy sobre los datos expandidos
#            (cada persona repetida tantas veces como indica su peso). Cuyo, 2017.
m = base_p21[base_p21['ANO4'].eq(2017) & base_p21['region'].eq('Cuyo')]
educ = (m['nivel_educativo'].cat.codes + 1).to_numpy()
expandido_educ = np.repeat(educ, m['PONDIIO'])
expandido_ing = np.repeat(m['p21_real'].to_numpy(), m['PONDIIO'])
print('c) SciPy expandido:', round(spearmanr(expandido_educ, expandido_ing).statistic, 6),
      f'({len(expandido_ing):,} filas)')
print('   Propia ponderada:', round(spearman_ponderada(educ, m['p21_real'], m['PONDIIO']), 6))

a) Pearson : 0.725 (esperado: 0.725)
   Spearman: 1.0 (esperado: 1.0)
b) SciPy   : 0.318914
   Propia  : 0.318914
c) SciPy expandido: 0.308948 (636,731 filas)
   Propia ponderada: 0.308948


Los tres controles coinciden con seis decimales. La función ponderada es correcta.

### 12.2 Correlación por año
Se calcula la Spearman ponderada entre el nivel educativo (1 a 6) y el indicador central para cada año.

In [24]:
def corr_educ_ingreso(g):
    educ = g['nivel_educativo'].cat.codes + 1        # 1 = primario incompleto ... 6 = superior completo
    return spearman_ponderada(educ, g['p21_real'], g['PONDIIO'])

corr_anual = (base_p21.groupby('ANO4')
                      .apply(corr_educ_ingreso, include_groups=False)
                      .rename('spearman_educ_p21'))
corr_anual.to_csv(RESULTADOS / 'correlacion_educ_ingreso_anual.csv')
print('Guardado:', (RESULTADOS / 'correlacion_educ_ingreso_anual.csv').exists())
corr_anual.round(3)

Guardado: True


,spearman_educ_p21
ANO4,
2017,0.340
2018,0.352
2019,0.372
2020,0.389
2021,0.399
2022,0.370
2023,0.373
2024,0.368
2025,0.343


La correlación es moderada y positiva en todos los años: sube de 0,340 en 2017 a 0,399 en 2021 y vuelve a 0,335 en 2026. Un mayor nivel educativo se asocia con un ingreso más alto, sin que eso implique causalidad. Solo se analiza la educación: la edad tiene una relación en forma de U invertida con el ingreso y la región es una variable sin orden, así que la correlación de Spearman no corresponde.

### 12.3 Correlación descartada
La celda siguiente se conserva solo como registro de una decisión. **Su resultado no se interpreta.**

In [25]:
# Correlación DESCARTADA: se conserva solo como registro de la decisión. No se interpreta.
# Son 10 puntos (años) y las dos series comparten una tendencia en el tiempo.
espuria = anual['tasa_desempleo'].corr(anual['p21_media'], method='spearman')
print('Spearman desempleo vs. media de p21_real entre años (n = 10):', round(espuria, 3))

Spearman desempleo vs. media de p21_real entre años (n = 10): 0.515


La correlación entre la tasa de desempleo y el ingreso medio a lo largo de los años da 0,515, pero es espuria: se calcula con solo 10 puntos y las dos series comparten una tendencia en el tiempo. No permite concluir ninguna relación entre desempleo e ingreso.

## 13. Sensibilidad al deflactor
El deflactor usa el promedio del IPC de marzo a mayo porque la EPH pregunta por el ingreso del mes anterior a la entrevista, y la base no informa en qué mes se hizo cada entrevista. Esta sección mide cuánto cambian los resultados con otros criterios razonables.

### 13.1 Factores con cada criterio
Se calculan los factores con cuatro criterios: marzo a mayo (el elegido), solo marzo, solo mayo y abril a junio. El mes base es siempre octubre de 2026.

In [26]:
# Lectura del IPC: mismos parámetros que en 01_preparacion
ipc = pd.read_csv(ARCHIVO_IPC, sep=';', encoding='latin-1', decimal=',')
ipc = ipc[ipc['Descripcion'].eq('NIVEL GENERAL')].copy()   # nivel general de cada región
ipc['anio'] = ipc['Periodo'] // 100
ipc['mes'] = ipc['Periodo'] % 100

MES_BASE = 202610                                          # octubre de 2026
ESCENARIOS = {
    'mar-may (elegido)': [3, 4, 5],
    'solo marzo': [3],
    'solo mayo': [5],
    'abr-jun': [4, 5, 6],
}

def factores(serie, meses):
    """Factor de cada año: IPC del mes base / promedio del IPC en los meses indicados."""
    base = serie.loc[serie['Periodo'].eq(MES_BASE), 'Indice_IPC'].iloc[0]
    promedio = serie[serie['mes'].isin(meses) & serie['anio'].between(2017, 2026)].groupby('anio')['Indice_IPC'].mean()
    return base / promedio

ipc_nacional = ipc[ipc['Region'].eq('Nacional')]
tabla_factores = pd.DataFrame({nombre: factores(ipc_nacional, meses) for nombre, meses in ESCENARIOS.items()})

# Inflación mensual del 2.º trimestre de 2024, el tramo más sensible
print('Inflación mensual 2024 (%):',
      ipc_nacional.loc[ipc_nacional['anio'].eq(2024) & ipc_nacional['mes'].between(3, 6),
                       ['mes', 'v_m_IPC']].set_index('mes')['v_m_IPC'].to_dict())
tabla_factores.round(3)

Inflación mensual 2024 (%): {3: 11.0, 4: 8.8, 5: 4.2, 6: 4.6}


,mar-may (elegido),solo marzo,solo mayo,abr-jun
anio,,,,
2017,117.790,120.454,115.678,115.764
2018,93.682,96.059,91.597,91.080
2019,60.068,62.081,58.232,58.281
2020,41.219,41.845,40.602,40.504
2021,28.241,29.335,27.278,27.282
2022,17.873,18.913,16.977,16.949
2023,8.539,9.257,7.924,7.957
2024,2.222,2.387,2.105,2.101
2025,1.495,1.531,1.467,1.467


Los criterios se separan más en los años de inflación mensual alta. En 2024, la inflación fue del 11,0 % en marzo, el 8,8 % en abril y el 4,2 % en mayo, así que el mes que se elija cambia el factor.

### 13.2 Efecto sobre la media nacional
Se recalcula la media del indicador central con cada criterio a partir del ingreso nominal (P21) de los mismos casos. El primer control confirma que el criterio elegido reproduce la sección 7.

In [27]:
# Media de p21 en pesos de octubre de 2026 con cada escenario.
# Se parte del ingreso nominal (P21) de los mismos casos del indicador central.
medias = {}
for nombre in ESCENARIOS:
    real = base_p21['P21'] * base_p21['ANO4'].map(tabla_factores[nombre])
    medias[nombre] = (pd.DataFrame({'real': real, 'w': base_p21['PONDIIO'], 'ANO4': base_p21['ANO4']})
                        .groupby('ANO4')
                        .apply(lambda g: media_ponderada(g['real'], g['w']), include_groups=False))
sensibilidad = pd.DataFrame(medias)

# Control: el escenario elegido debe reproducir la media de la sección 7
print('Reproduce p21_anual:', np.allclose(sensibilidad['mar-may (elegido)'], p21_anual['media']))

sensibilidad.to_csv(RESULTADOS / 'sensibilidad_deflactor.csv')
print('Guardado:', (RESULTADOS / 'sensibilidad_deflactor.csv').exists())

# Variaciones clave con cada escenario
variaciones = pd.DataFrame({
    '2024 vs 2023 (%)': (sensibilidad.loc[2024] / sensibilidad.loc[2023] - 1) * 100,
    '2025 vs 2024 (%)': (sensibilidad.loc[2025] / sensibilidad.loc[2024] - 1) * 100,
    '2026 vs 2017 (%)': (sensibilidad.loc[2026] / sensibilidad.loc[2017] - 1) * 100,
    'año mínimo': sensibilidad.idxmin(),
})
variaciones.round(1)

Reproduce p21_anual: True
Guardado: True


,2024 vs 2023 (%),2025 vs 2024 (%),2026 vs 2017 (%),año mínimo
mar-may (elegido),-12.6,30.3,-16.1,2024
solo marzo,-13.4,24.2,-15.9,2024
solo mayo,-10.8,35.0,-16.5,2024
abr-jun,-11.3,35.2,-16.5,2024


- **2024 es el mínimo de la serie con los cuatro criterios.**
- **La caída de 2024 frente a 2023** va de −10,8 % a −13,4 %, y **la variación entre 2017 y 2026**, de −15,9 % a −16,5 %. Las dos conclusiones son estables.
- **El repunte de 2025 frente a 2024 es lo más sensible:** va de +24,2 % a +35,2 %. Se informa como rango.

### 13.3 IPC nacional frente a IPC regional
Se recalcula la variación 2017–2026 de cada región con el IPC de su propia región (promedio de marzo a mayo) y se compara con la obtenida con el IPC nacional.

In [28]:
# Variación 2017–2026 de la media por región: IPC nacional frente al IPC de cada región (marzo a mayo)
IPC_REGION = {'Gran Buenos Aires': 'GBA', 'Noroeste': 'Noroeste', 'Noreste': 'Noreste',
              'Cuyo': 'Cuyo', 'Pampeana': 'Pampeana', 'Patagonia': 'Patagonia'}

filas = []
for region_eph, region_ipc in IPC_REGION.items():
    g = base_p21[base_p21['region'].eq(region_eph) & base_p21['ANO4'].isin([2017, 2026])]
    f_reg = factores(ipc[ipc['Region'].eq(region_ipc)], [3, 4, 5])
    real_reg = g['P21'] * g['ANO4'].map(f_reg)
    medias_nac = g.groupby('ANO4').apply(lambda h: media_ponderada(h['p21_real'], h['PONDIIO']), include_groups=False)
    medias_reg = (pd.DataFrame({'x': real_reg, 'w': g['PONDIIO'], 'ANO4': g['ANO4']})
                    .groupby('ANO4').apply(lambda h: media_ponderada(h['x'], h['w']), include_groups=False))
    filas.append({
        'region': region_eph,
        'var_IPC_nacional_%': (medias_nac[2026] / medias_nac[2017] - 1) * 100,
        'var_IPC_regional_%': (medias_reg[2026] / medias_reg[2017] - 1) * 100,
    })

sens_regional = pd.DataFrame(filas).set_index('region')
sens_regional['diferencia_puntos'] = sens_regional['var_IPC_regional_%'] - sens_regional['var_IPC_nacional_%']
display(sens_regional.round(1))
print('Diferencia máxima (puntos):', round(sens_regional['diferencia_puntos'].abs().max(), 1))

,var_IPC_nacional_%,var_IPC_regional_%,diferencia_puntos
region,,,
Gran Buenos Aires,-14.1,-13.9,0.3
Noroeste,-16.8,-17.1,-0.3
Noreste,-27.6,-28.2,-0.7
Cuyo,-22.2,-23.1,-0.8
Pampeana,-16.5,-16.5,0.0
Patagonia,-18.1,-18.4,-0.3


Diferencia máxima (puntos): 0.8


Con el IPC regional, la variación de cada región cambia como máximo 0,8 puntos, y el orden entre regiones no se altera. El IPC nacional no refleja las diferencias de costo de vida entre regiones, así que los niveles de ingreso entre regiones se comparan con esa salvedad.

## 14. Resumen y limitaciones

### Resultados principales (descriptivos)
- El ingreso real de la ocupación principal cae un 16,1 % entre 2017 y 2026 con la media (−13,9 % con la mediana), con un mínimo en 2024.
- Por etapa, el ingreso medio baja de 1,45 millones antes a 1,30 millones durante y a 1,22 millones después de la cuarentena. En cambio, el empleo y el desempleo de la etapa "después" están mejor que antes.
- El ingreso crece con el nivel educativo en todos los años, y la brecha entre superior completo y primario incompleto se amplía de 2,56 a 2,83 veces.
- Noreste es la región que más cae y Patagonia la de mayor ingreso.
- La correlación entre educación e ingreso es moderada (entre 0,34 y 0,40).

### Solidez de los resultados
- Las funciones propias coinciden con numpy y SciPy.
- La tasa de desempleo reproduce la oficial del INDEC con una diferencia máxima de 0,1 puntos.
- Las conclusiones principales se mantienen sin el 1 % de ingresos más altos y con los cuatro criterios de deflactor.

### Limitaciones
- **Relevamiento no presencial en 2020 y 2021:** las cifras de esos años combinan el efecto de la pandemia con el del cambio de modalidad.
- **No respuesta del ingreso:** ponderada, va del 18,0 % al 25,3 % de los ocupados, y se concentra en Gran Buenos Aires.
- **Agrupamiento de los ingresos declarados en montos redondos:** afecta a la mediana y a los percentiles. Las variaciones no robustas se informan con la media.
- **Sin errores estándar ni pruebas de significación:** la precisión se evalúa solo por la cantidad de casos. Las diferencias chicas entre grupos pueden no ser significativas.
- **Deflactor:** el promedio de marzo a mayo es una aproximación. El repunte de 2025 se informa como rango (+24 % a +35 %).
- **IPC nacional:** no refleja las diferencias de costo de vida entre regiones.
- **65 años o más:** es el grupo menos preciso y sus resultados no son concluyentes.
- **Base preliminar del 2.º trimestre de 2026** y **cobertura** de los 31 aglomerados urbanos, sin la población rural.

### Próximo paso
`03_visualizaciones.ipynb` usa las 13 tablas de `resultados/`. No necesita volver a leer la base.